# CONVOLUTIONAL 1D AE TRAINING

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import h5py

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

tf.random.set_seed(42)
np.random.seed(42)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

In [ ]:
BACKGROUND_FILE = "../data/datasets/convolutional/background_convolutional_dataset.h5"

with h5py.File(BACKGROUND_FILE, "r") as f:
    X_train = f["X_train"][:].astype(np.float32)
    X_val = f["X_val"][:].astype(np.float32)
    X_test = f["X_test"][:].astype(np.float32)

print(f'{"X_train":<10}: {X_train.shape}')
print(f'{"X_val":<10}: {X_val.shape}')
print(f'{"X_test":<10}: {X_test.shape}')

In [ ]:
CONV1D_MODELS_DIR = "../models/convolutional1d_AE"
os.makedirs(CONV1D_MODELS_DIR, exist_ok=True)

MODEL_NAME = "conv1d_ae_0"

SAVE_MODEL_PATH = os.path.join(CONV1D_MODELS_DIR, f"{MODEL_NAME}.keras")

In [ ]:
N_OBJ = 19
N_FEAT = 3
PT_STD = X_train[:, :, 0].std() + 1e-8

W_CURR = {"pt": 1.0, "eta": 1.0, "phi": 0.5}

In [ ]:
def masked_trigger_loss(y_true, y_pred):
    y_true = tf.reshape(y_true, (-1, N_OBJ, N_FEAT))
    y_pred = tf.reshape(y_pred, (-1, N_OBJ, N_FEAT))

    mask = tf.cast(tf.not_equal(y_true[:, :, 0:1], 0.0), tf.float32)

    pt_true = y_true[:, :, 0:1]
    pt_pred = y_pred[:, :, 0:1]
    pt_loss = tf.square(pt_true - pt_pred) / (PT_STD ** 2)

    eta_true = y_true[:, :, 1:2]
    eta_pred = y_pred[:, :, 1:2]
    eta_loss = tf.square((eta_true - eta_pred) / 3.0)

    phi_true = y_true[:, :, 2:3]
    phi_pred = y_pred[:, :, 2:3]
    dphi = tf.atan2(tf.sin(phi_true - phi_pred), tf.cos(phi_true - phi_pred))
    phi_loss = tf.square(dphi)

    total = W_CURR["pt"] * pt_loss + W_CURR["eta"] * eta_loss + W_CURR["phi"] * phi_loss
    total *= mask

    n_present = tf.maximum(tf.reduce_sum(mask, axis=[1, 2]), 1.0)
    return tf.reduce_mean(tf.reduce_sum(total, axis=[1, 2]) / n_present)

In [ ]:
def build_conv1d_ae(input_shape=(19, 3), latent_dim=16, name="conv1d_ae"):
    inp = keras.Input(shape=input_shape, name="input")

    x = layers.BatchNormalization()(inp)
    x = layers.Conv1D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)

    x = layers.Conv1D(64, 3, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)

    x = layers.Conv1D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)

    x = layers.Flatten()(x)
    z = layers.Dense(latent_dim, name="latent")(x)

    x = layers.Dense(10 * 64, use_bias=False)(z)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Reshape((10, 64))(x)

    x = layers.UpSampling1D(2)(x)
    x = layers.Conv1D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)

    x = layers.Conv1D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)

    x = layers.Cropping1D(cropping=(0, 1))(x)
    out = layers.Conv1D(3, 1, padding="same", name="reco")(x)

    return Model(inp, out, name=name)

conv1d_ae = build_conv1d_ae(name=MODEL_NAME)
conv1d_ae.summary()

In [ ]:
EPOCHS = 100
BATCH_SIZE = 1024
LR = 1e-3

conv1d_ae.compile(optimizer=keras.optimizers.Adam(LR), loss=masked_trigger_loss)

cb = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, verbose=1),
    keras.callbacks.ModelCheckpoint(SAVE_MODEL_PATH, save_best_only=True),
]

history_c1d = conv1d_ae.fit(
    X_train,
    X_train,
    validation_data=(X_val, X_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cb,
    verbose=1,
)

In [ ]:
def plot_history(history, title="Training history"):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.semilogy(history.history["loss"], label="Train")
    ax.semilogy(history.history["val_loss"], label="Validation")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (log scale)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_history(history_c1d, "Convolutional 1D AE")

In [ ]:
def compute_masked_components_arrays(model, X, batch_size=8192):
    y_true = tf.convert_to_tensor(X, dtype=tf.float32)
    y_pred = tf.convert_to_tensor(model.predict(X, batch_size=batch_size, verbose=0), dtype=tf.float32)

    y_true = tf.reshape(y_true, (-1, N_OBJ, N_FEAT))
    y_pred = tf.reshape(y_pred, (-1, N_OBJ, N_FEAT))

    mask = tf.cast(tf.not_equal(y_true[:, :, 0:1], 0.0), tf.float32)
    n_present = tf.maximum(tf.reduce_sum(mask, axis=[1, 2]), 1.0)

    pt_true = y_true[:, :, 0:1]
    pt_pred = y_pred[:, :, 0:1]
    pt_loss = tf.square(pt_true - pt_pred) / (PT_STD ** 2)

    eta_true = y_true[:, :, 1:2]
    eta_pred = y_pred[:, :, 1:2]
    eta_loss = tf.square((eta_true - eta_pred) / 3.0)

    phi_true = y_true[:, :, 2:3]
    phi_pred = y_pred[:, :, 2:3]
    dphi = tf.atan2(tf.sin(phi_true - phi_pred), tf.cos(phi_true - phi_pred))
    phi_loss = tf.square(dphi)

    pt_evt = tf.reduce_sum(pt_loss * mask, axis=[1, 2]) / n_present
    eta_evt = tf.reduce_sum(eta_loss * mask, axis=[1, 2]) / n_present
    phi_evt = tf.reduce_sum(phi_loss * mask, axis=[1, 2]) / n_present

    return {
        "pt_evt": pt_evt.numpy(),
        "eta_evt": eta_evt.numpy(),
        "phi_evt": phi_evt.numpy(),
    }

def summarize_components(comp, weights):
    pt_m = float(np.mean(comp["pt_evt"]))
    eta_m = float(np.mean(comp["eta_evt"]))
    phi_m = float(np.mean(comp["phi_evt"]))

    pt_w = weights["pt"] * pt_m
    eta_w = weights["eta"] * eta_m
    phi_w = weights["phi"] * phi_m
    tot_w = pt_w + eta_w + phi_w

    return {
        "unweighted_mean": {"pt": pt_m, "eta": eta_m, "phi": phi_m},
        "weighted_mean": {"pt": pt_w, "eta": eta_w, "phi": phi_w, "total": tot_w},
        "weighted_fraction": {
            "pt": pt_w / (tot_w + 1e-12),
            "eta": eta_w / (tot_w + 1e-12),
            "phi": phi_w / (tot_w + 1e-12),
        },
    }

def print_summary(split_name, summary):
    print(f"\n=== {split_name} ===")
    print("Unweighted means:")
    print(
        f"  pt={summary['unweighted_mean']['pt']:.6f}, "
        f"eta={summary['unweighted_mean']['eta']:.6f}, "
        f"phi={summary['unweighted_mean']['phi']:.6f}"
    )
    print("Weighted means:")
    print(
        f"  pt={summary['weighted_mean']['pt']:.6f}, "
        f"eta={summary['weighted_mean']['eta']:.6f}, "
        f"phi={summary['weighted_mean']['phi']:.6f}, "
        f"total={summary['weighted_mean']['total']:.6f}"
    )
    print("Weighted fractions:")
    print(
        f"  pt={summary['weighted_fraction']['pt']:.3f}, "
        f"eta={summary['weighted_fraction']['eta']:.3f}, "
        f"phi={summary['weighted_fraction']['phi']:.3f}"
    )

comp_train = compute_masked_components_arrays(conv1d_ae, X_train)
comp_val = compute_masked_components_arrays(conv1d_ae, X_val)

summary_train = summarize_components(comp_train, W_CURR)
summary_val = summarize_components(comp_val, W_CURR)

print_summary("TRAIN", summary_train)
print_summary("VAL", summary_val)